# SPARK Academy 2026 | Introduction to Machine Learning
## Notebook 2: Unsupervised Learning

**Instructor:** Aondona Moses
**Dataset:** Pima Indians Diabetes Dataset

---

### What you will learn:
1. What is Unsupervised Learning?
2. **K-Means Clustering** | grouping patients by similarity
3. Finding the optimal number of clusters (Elbow Method, Silhouette Score)
4. **Hierarchical Clustering** | building a tree of patient groups
5. Comparing clusters to actual diabetes labels
6. When to use clustering in medical research

---

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Load and clean
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
df = pd.read_csv(url, names=columns)

zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan).fillna(df[col].median())

# Separate features and labels
X = df.drop("Outcome", axis=1)
y_true = df["Outcome"]  # we will use this ONLY to evaluate clusters, not to train

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data loaded: {X.shape[0]} patients, {X.shape[1]} features")
print(f"Actual diabetes distribution: {y_true.value_counts().to_dict()}")

## 2. What is Unsupervised Learning?

In **supervised learning**, we have labels (diabetic / not diabetic) and teach the model to predict them.

In **unsupervised learning**, we have **no labels**. We ask the algorithm to **find patterns on its own**.

### Why is this useful in medicine?
- **Discover patient subtypes** — maybe there are 3 types of diabetic patients, not just 2
- **Anomaly detection** — find unusual patients who need attention
- **Data exploration** — understand structure before building supervised models
- **When labels are expensive** — getting a radiologist to label 10,000 scans is costly

## 3. K-Means Clustering

K-Means groups data into **K clusters** by minimizing the distance from each point to its cluster center.

**Algorithm:**
1. Choose K (number of clusters)
2. Randomly place K cluster centers
3. Assign each patient to the nearest center
4. Move each center to the mean of its assigned patients
5. Repeat steps 3-4 until stable

In [ ]:
# K-Means with K=2 (matching our binary diabetes outcome)
kmeans_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters_2 = kmeans_2.fit_predict(X_scaled)

print("=== K-Means (K=2) ===")
print(f"Cluster sizes: {np.bincount(clusters_2)}")
print(f"Cluster centers shape: {kmeans_2.cluster_centers_.shape}")

# Compare to actual diabetes labels
ari = adjusted_rand_score(y_true, clusters_2)
print(f"\nAdjusted Rand Index: {ari:.4f}")
print("(ARI measures agreement between clusters and true labels)")
print("(1.0 = perfect match, 0.0 = random)")

In [ ]:
# Visualize clusters using PCA (reduce to 2D for plotting)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_2, cmap="RdYlBu",
                           alpha=0.6, s=30, edgecolors="white", linewidths=0.3)
axes[0].set_title("K-Means Clusters (K=2)", fontsize=14, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.colorbar(scatter1, ax=axes[0], label="Cluster")

# Actual labels
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap="RdYlBu",
                           alpha=0.6, s=30, edgecolors="white", linewidths=0.3)
axes[1].set_title("Actual Diabetes Labels", fontsize=14, fontweight="bold")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.colorbar(scatter2, ax=axes[1], label="Outcome")

plt.tight_layout()
plt.show()

In [ ]:
# What do the clusters look like?
df_clustered = df.copy()
df_clustered["Cluster"] = clusters_2

print("=== Cluster Profiles ===")
print(df_clustered.groupby("Cluster")[X.columns].mean().round(2).T)
print("\n=== Diabetes rate per cluster ===")
print(df_clustered.groupby("Cluster")["Outcome"].mean().round(3))

### 3.1 Finding the Optimal K | Elbow Method & Silhouette Score

In [ ]:
# Elbow method + Silhouette score
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(K_range, inertias, "bo-", linewidth=2, markersize=8)
axes[0].set_xlabel("Number of Clusters (K)", fontsize=12)
axes[0].set_ylabel("Inertia (within-cluster sum of squares)", fontsize=12)
axes[0].set_title("Elbow Method", fontsize=14, fontweight="bold")

# Silhouette
axes[1].plot(K_range, silhouettes, "ro-", linewidth=2, markersize=8)
axes[1].set_xlabel("Number of Clusters (K)", fontsize=12)
axes[1].set_ylabel("Silhouette Score", fontsize=12)
axes[1].set_title("Silhouette Score", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Best K by silhouette score: {best_k}")
print(f"Silhouette scores: {dict(zip(K_range, [f'{s:.3f}' for s in silhouettes]))}")

## 4. Hierarchical Clustering

Hierarchical clustering builds a **tree (dendrogram)** of nested clusters. Unlike K-means, you do not need to specify K in advance.

**Two approaches:**
- **Agglomerative** (bottom-up): Start with each patient as a cluster, merge the closest pairs
- **Divisive** (top-down): Start with one cluster, split recursively

We use **agglomerative** (more common).

In [ ]:
# Compute linkage matrix for dendrogram
# Using a subset for visualization (dendrogram with 768 patients is unreadable)
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=100, replace=False)
X_sample = X_scaled[sample_idx]
y_sample = y_true.values[sample_idx]

linkage_matrix = linkage(X_sample, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(linkage_matrix, truncate_mode="lastp", p=20,
           leaf_rotation=90, leaf_font_size=10,
           show_contracted=True)
plt.title("Hierarchical Clustering Dendrogram (100 patient sample)", fontsize=14, fontweight="bold")
plt.xlabel("Cluster Size", fontsize=12)
plt.ylabel("Distance", fontsize=12)
plt.axhline(y=25, color="r", linestyle="--", label="Cut at 2 clusters")
plt.legend()
plt.tight_layout()
plt.show()
print("The red dashed line shows where we would cut to get 2 clusters")

In [ ]:
# Agglomerative clustering on full dataset
agg_2 = AgglomerativeClustering(n_clusters=2, linkage="ward")
agg_clusters = agg_2.fit_predict(X_scaled)

print("=== Agglomerative Clustering (K=2) ===")
print(f"Cluster sizes: {np.bincount(agg_clusters)}")

ari_agg = adjusted_rand_score(y_true, agg_clusters)
print(f"Adjusted Rand Index: {ari_agg:.4f}")

In [ ]:
# Compare all three: K-Means, Hierarchical, Actual
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, labels, title in zip(axes,
    [clusters_2, agg_clusters, y_true],
    ["K-Means (K=2)", "Hierarchical (K=2)", "Actual Labels"]):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="RdYlBu",
                        alpha=0.6, s=30, edgecolors="white", linewidths=0.3)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles comparison
df_agg = df.copy()
df_agg["Cluster"] = agg_clusters

print("=== Hierarchical Cluster Profiles ===")
print(df_agg.groupby("Cluster")[X.columns].mean().round(2).T)
print("\n=== Diabetes rate per cluster ===")
print(df_agg.groupby("Cluster")["Outcome"].mean().round(3))

## 5. K-Means vs Hierarchical | When to Use Which?

| Feature | K-Means | Hierarchical |
|---------|---------|-------------|
| Speed | Fast (good for large datasets) | Slower (computes all pairwise distances) |
| K required? | Yes, must specify K | No, cut the dendrogram at any level |
| Shape | Finds spherical clusters | Can find irregular shapes |
| Reproducibility | Depends on initialization | Deterministic |
| Visualization | Scatter plots | Dendrogram (shows structure) |
| Best for | Large datasets, clear groups | Small-medium data, exploring hierarchy |

## 6. Key Takeaways

1. **Unsupervised learning** finds patterns without labels — useful for discovering patient subtypes.
2. **K-Means** is fast and simple but requires choosing K. Use the elbow method and silhouette score.
3. **Hierarchical clustering** gives you a dendrogram — you can see the full structure and cut at any level.
4. **PCA** reduces dimensions for visualization when you have many features.
5. Clusters do not perfectly match disease labels — that is expected! Clusters find structure the labels might miss.
6. In medical research, clustering can reveal **subtypes** that traditional diagnosis categories miss.

### What’s Next:
- **Notebook 3:** Full ML Implementation Pipeline with Multiple Models

---
*SPARK Academy 2026 — Train for Change, From Science to Practice* 🚀